# Session 4 lab — Relational data management and SQL II

The same six tables as last week, finally connected. You will answer four
business questions that each need a join, and rank trips within each driver with
a window function. Part C, *optional*, reads the plan PostgreSQL chooses for one
of your queries, before and after an index.

To hand in: your four queries with their output, and the row counts asked for in
question 4 with your corrected answer.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import psycopg


def find(*candidates: str) -> Path:
    """This notebook runs from its own folder, or from anywhere in the container."""
    for candidate in candidates:
        if Path(candidate).exists():
            return Path(candidate).resolve()
    raise SystemExit(f"not found: {candidates[0]}")


SQL = find("sql", "session-04/sql", "/app/labs/session-04/sql")
SCHEMA = find(
    "../session-03/sql/schema.sql",
    "session-03/sql/schema.sql",
    "/app/labs/session-03/sql/schema.sql",
)
CSV = find(
    "../../data/nyc-taxi-normalized",
    "data/nyc-taxi-normalized",
    "/app/data/nyc-taxi-normalized",
)

DB = psycopg.connect("host=postgres port=5432 user=labs password=labs dbname=labs")
DB.autocommit = True
DB.execute("SET TIME ZONE 'America/New_York'")


def sql(statement: str) -> pd.DataFrame:
    """Run one statement, and show whatever rows come back."""
    if not statement.strip():
        print("nothing to run yet — write your query between the quotes")
        return pd.DataFrame()
    with DB.cursor() as cursor:
        cursor.execute(statement)
        if cursor.description is None:
            return pd.DataFrame()
        columns = [column.name for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)


def explain(query: str) -> None:
    """Print the plan PostgreSQL would choose for a query, without running it."""
    if not query.strip():
        print("nothing to explain yet — paste your query between the quotes")
        return
    for line in sql(f"EXPLAIN {query}").iloc[:, 0]:
        print(line)


print(f"schema:       {SCHEMA}")
print(f"this session: {SQL}")
print(f"data to load: {CSV}")

## Setting up (2 min)

The cell below rebuilds the database from session 3's `schema.sql`, with one
change for this session, in `sql/split_fares.sql`. Read that file: it is three
lines of comment and one statement. Since January a rider can split a fare across
several cards, so a trip can now have more than one payment.

It also loads twelve drivers hired in the last days of January.

**Run it again at any point** — it drops everything, including any index you
create in part C, and starts over.

In [ ]:
TABLES = ["zones", "drivers", "riders", "driver_balances", "trips", "payments"]

# Session 4 loads the payments in which some fares were split.
FILES = {table: table for table in TABLES} | {"payments": "payments_split"}


def reset_database() -> None:
    """Create the six tables, apply this session's change, and load the data."""
    DB.execute(SCHEMA.read_text())
    DB.execute((SQL / "split_fares.sql").read_text())

    for table in TABLES:
        with (CSV / f"{FILES[table]}.csv").open() as file, DB.cursor() as cursor:
            copy_in = f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER)"
            with cursor.copy(copy_in) as copy:
                while chunk := file.read(1 << 20):
                    copy.write(chunk)

    # Fresh statistics, so the planner in part C knows how big each table is.
    DB.execute("ANALYZE")

    counts = ", ".join(
        f"{table} {sql(f'SELECT count(*) AS n FROM {table}').at[0, 'n']:,}"
        for table in TABLES
    )
    print(f"loaded: {counts}")


started = time.perf_counter()
reset_database()
print(f"in {time.perf_counter() - started:.1f} s")

## Part A — Four questions, four joins (12 min)

One statement per question, in the empty cell under it.

If you would rather build and test a query in pgAdmin, at
<http://localhost:5050> (password `labs`), open *Servers &rarr; labs &rarr;
Databases &rarr; labs*, then *Tools &rarr; Query Tool*. It runs against the same
database. Paste the finished query back here, so it is handed in with its output.

From now on, give every table an alias (`trips t`, `drivers d`) and prefix every
column with it. With two tables it is tidy; with three it is the only way to read
the query back.

### 1. Trips and fares by borough

For each borough, the number of trips picked up there and the average fare,
busiest borough first.

`trips` knows the pickup zone; `zones` knows which borough a zone is in.

In [ ]:
sql("""

""")

### 2. January fares for every driver

Payroll needs one row for **every driver on the books**: the driver id, their
name, the number of trips they drove in January, and the total of those trips'
fares. Lowest total first.

A driver who drove nothing belongs on the list with 0 trips and 0.00 in fares —
not with a blank, and not with a 1.

In [ ]:
sql("""

""")

How many rows does your answer have, and how many rows are in `drivers`?

In [ ]:
sql("""

""")

### 3. Who works the airports

The ten drivers who earned the most in fares from **airport pickups** — trips
whose pickup zone has `service_zone = 'Airports'`. For each: name, licence
number, the number of airport pickups, and the total fare from them.

In [ ]:
sql("""

""")

### 4. Revenue by payment method

Finance is closing January. For each payment method, they want the number of
trips paid that way, the total fare of those trips, and the total amount
collected. Largest total fare first.

In [ ]:
sql("""

""")

Before this goes to finance, count rows. How many rows are in `trips`? How many
rows does `trips` joined to `payments` produce, before any `GROUP BY`?

In [ ]:
sql("""

""")

In [ ]:
sql("""

""")

Is the answer you gave in question 4 right? Say how you know, in one or two
sentences. If it is not, write the query that is, below.

*Your sentences:*

In [ ]:
sql("""

""")

## Part B — Top three per driver (8 min)

For every driver, their three best-tipped trips of January: the driver's name, the
trip id, when it was picked up, and the tip. Ordered by driver, then biggest tip
first.

Use `row_number() OVER (PARTITION BY ... ORDER BY ...)` inside a `WITH`, and
filter the result outside it. Before you hand it in, look at the tips that come
back.

In [ ]:
sql("""

""")

How many rows came back, and how many should have?

In [ ]:
sql("""

""")

### Optional: the same question with `GROUP BY` only

If you have time, try it without a window function. `GROUP BY` gives you each
driver's biggest tip easily. The second-biggest is harder. If a query runs for
more than a minute, interrupt the kernel — that is part of the answer.

In [ ]:
sql("""

""")

## Part C — Read the plan, then change it (10 min, optional)

For anyone with time left, or afterwards. Nothing later depends on it.

Take your query from **question 2** and ask PostgreSQL how it would run it.
`explain()` prints the plan without running the query.

Three things to read in the plan, as in the lecture: the operation on each table
(`Seq Scan`, `Index Scan`), the number of rows it expects, and the cost.

Two words you will see and can read past. `Hash Right Join` is your `LEFT JOIN`
with the two tables swapped, which is all a right join is. `Gather` and `Parallel
Seq Scan` mean the scan is split across several workers; it is still a scan.

In [ ]:
explain("""

""")

### Your prediction, before you create anything

Would an index on `trips (driver_id)` change this plan? Say yes or no, and why,
in a sentence or two.

A prediction that the index will **not** help is a perfectly good prediction. You
are graded on nothing here, and you are certainly not graded on whether the plan
changed — only on whether you said what would happen and why.

*Your prediction:*

Now the same question for **one driver**: copy your query, add
`WHERE d.driver_id = 42` (or whichever alias you gave `drivers`), and explain it.
Then predict again: would the same index change *this* plan?

*Your prediction:*

In [ ]:
explain("""

""")

Create the index.

In [ ]:
sql("CREATE INDEX ON trips (driver_id)")

Explain both queries again.

In [ ]:
explain("""

""")

In [ ]:
explain("""

""")

Were your two predictions right? For each, say what changed in the plan, or what
did not, and why.

On the one-driver plan you may see a `Bitmap Index Scan` feeding a `Bitmap Heap
Scan`. That is the index being used; how it works in detail is not today's
subject.

*Your answer:*